In [1]:
import argparse

import matplotlib.pyplot as plt

import rng_control
import mnist_train
import pandas as pd
import json
from pathlib import Path

def load_results_from_json(filename):
    with open(filename, 'r') as f:
        return json.load(f)


results = load_results_from_json("results_fashion/results.json")

In [3]:
import matplotlib.pyplot as plt
import numpy as np

def plot_losses(results, ema_alpha=0.05, raw_alpha=0.08, band_alpha=0.22):
    plt.style.use('default')
    engines = list(results.keys())
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    eps = 1e-12

    def ema(x, alpha=0.05):
        x = np.asarray(x, dtype=float)
        if x.size == 0:
            return x
        y = np.empty_like(x, dtype=float)
        y[0] = x[0]
        for i in range(1, len(x)):
            y[i] = alpha * x[i] + (1.0 - alpha) * y[i - 1]
        return y

    for i, engine in enumerate(engines):
        fig, ax = plt.subplots(figsize=(6, 4))
        seeds_data = results.get(engine, {})
        color = colors[i % len(colors)]

        smoothed_runs = []

        for seed, data in seeds_data.items():
            if "losses" not in data:
                continue

            series = np.asarray(data["losses"], dtype=float)
            series = series[np.isfinite(series)]
            if series.size == 0:
                continue

            series = np.clip(series, eps, None)

            ax.plot(series, color=color, linewidth=0.8, alpha=raw_alpha)

            smooth = ema(series, alpha=ema_alpha)
            smooth = np.clip(smooth, eps, None)
            smoothed_runs.append(smooth)

        if smoothed_runs:
            min_len = min(len(s) for s in smoothed_runs)
            runs = np.stack([s[:min_len] for s in smoothed_runs], axis=0)

            x = np.arange(min_len)

            median = np.median(runs, axis=0)
            q25 = np.percentile(runs, 25, axis=0)
            q75 = np.percentile(runs, 75, axis=0)
            q10 = np.percentile(runs, 10, axis=0)
            q90 = np.percentile(runs, 90, axis=0)

            # Variance across seeds at each aligned step
            step_var = np.var(runs, axis=0)
            mean_var = float(np.mean(step_var))
            final_var = float(step_var[-1])

            ax.fill_between(
                x, q10, q90,
                color=color,
                alpha=band_alpha * 0.45,
                linewidth=0
            )
            ax.fill_between(
                x, q25, q75,
                color=color,
                alpha=band_alpha,
                linewidth=0
            )
            ax.plot(x, median, color=color, linewidth=0.5)

            ax.text(
                0.02, 0.98,
                f"Var (mean): {mean_var:.3e}\nVar (final): {final_var:.3e}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=9,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
            )

        ax.set_yscale('log')
        ax.set_xlabel("Steps")
        ax.set_ylabel("Loss")
        ax.set_title(engine)
        ax.grid(True, alpha=0.3)
        ax.margins(x=0)

        plt.tight_layout()
        plt.savefig(f"{engine}_losses.png", dpi=300, bbox_inches="tight")
        plt.close(fig)


def plot_metric_per_engine(results, metric="losses"):
    plt.style.use('default')
    engines = list(results.keys())
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for i, engine in enumerate(engines):
        fig, ax = plt.subplots(figsize=(5, 4))
        seeds_data = results.get(engine, {})

        if seeds_data:
            for seed, data in seeds_data.items():
                if metric in data:
                    series = data[metric]
                    # Plot the line
                    ax.plot(series, color=colors[i % len(colors)], linewidth=1.5, alpha=0.5)

                    # Add label for the seed at the last data point
                    ax.annotate(str(seed),
                                xy=(len(series) - 1, series[-1]),
                                xytext=(5, 0),
                                textcoords='offset points',
                                fontsize=8,
                                color=colors[i % len(colors)])

            if metric == "losses":
                ax.set_yscale('log')
                ax.set_xlabel("Steps")
            else:
                ax.set_xlabel("Epochs")

            ax.set_title(engine)
            ax.set_ylabel(metric.replace('_', ' ').capitalize())
            ax.grid(True, alpha=0.3)
            plt.tight_layout()

            plt.savefig(f"{engine}_{metric}.png", dpi=300)
            plt.close(fig)

plot_losses(results)
plot_metric_per_engine(results, metric="test_losses")
plot_metric_per_engine(results, metric="accuracies")
